In [1]:
pip install gensim underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import multiprocessing


import csv
import sys

csv.field_size_limit(sys.maxsize)

class ArxivCSVCorpus:
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __iter__(self):
        for file_path in self.file_paths:
            with open(file_path, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    abstract_text = row.get('abstracts', '')
                    if abstract_text and abstract_text.strip():
                        yield simple_preprocess(abstract_text, deacc=True)

def main():
    dataset_paths = [
        '/kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts/arxiv_data.csv', 
        '/kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts/arxiv_data_210930-054931.csv'
    ]
    
    sentences = ArxivCSVCorpus(dataset_paths)
    
    cores = multiprocessing.cpu_count()

    model = Word2Vec(
        sentences=sentences, 
        vector_size=200,      # Kích thước vector nhúng (embedding size)
        window=5,             # Kích thước cửa sổ ngữ cảnh (context window)
        min_count=10,         # Bỏ qua các từ xuất hiện ít hơn 10 lần trong toàn bộ corpus
        workers=cores,        # Số luồng xử lý
        sg=1,                 # sg=1 là Skip-gram, sg=0 là CBOW
        epochs=5              # Số vòng lặp qua toàn bộ dataset
    )
    
    model_path = 'arxiv_word2vec.model'
    model.save(model_path)
    print(f"Đã lưu mô hình tại: {model_path}")
        
    test_words = ['convolutional', 'quantum', 'transformer', 'topology']
    
    for word in test_words:
        print(f"\nCác từ có vector gần nhất với '{word}':")
        similar_words = model.wv.most_similar(word, topn=5)
        for sim_word, score in similar_words:
            print(f" - {sim_word} (độ tương đồng: {score:.4f})")

if __name__ == "__main__":
    main()

Đã lưu mô hình tại: arxiv_word2vec.model

Các từ có vector gần nhất với 'convolutional':
 - convolution (độ tương đồng: 0.7123)
 - fcnn (độ tương đồng: 0.6974)
 - fnn (độ tương đồng: 0.6406)
 - deconvolutional (độ tương đồng: 0.6402)
 - cnn (độ tương đồng: 0.6343)

Các từ có vector gần nhất với 'quantum':
 - mechanics (độ tương đồng: 0.5241)
 - chemistry (độ tương đồng: 0.5091)
 - optics (độ tương đồng: 0.5048)
 - qml (độ tương đồng: 0.5012)
 - integrators (độ tương đồng: 0.4705)

Các từ có vector gần nhất với 'transformer':
 - transformers (độ tương đồng: 0.6869)
 - xl (độ tương đồng: 0.5854)
 - vit (độ tương đồng: 0.5830)
 - longformer (độ tương đồng: 0.5715)
 - reformer (độ tương đồng: 0.5463)

Các từ có vector gần nhất với 'topology':
 - structure (độ tương đồng: 0.6333)
 - topological (độ tương đồng: 0.5947)
 - structures (độ tương đồng: 0.5613)
 - mesoscopic (độ tương đồng: 0.5323)
 - homeomorphic (độ tương đồng: 0.5031)
